# SmartAudit Lab on Kaggle
**Before you start:** right panel → *Session options* → **Internet: On**, **Accelerator: GPU T4 x2** (needed only for SLMs), and *Add-ons → Secrets*: add `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` (and attach them to this notebook).

## 1. Get the code
Upload `smartaudit-lab.zip` as a Kaggle **Dataset** and add it to this notebook (Add Input), then run the cell. Adjust the path if your dataset has another name.

In [ ]:
import glob, shutil, os, zipfile
src = glob.glob('/kaggle/input/*/smartaudit-lab*') + glob.glob('/kaggle/input/*/*.zip')
print(src)
if os.path.isdir('/kaggle/working/smartaudit-lab'): shutil.rmtree('/kaggle/working/smartaudit-lab')
for s in src:
    if s.endswith('.zip'):
        zipfile.ZipFile(s).extractall('/kaggle/working'); break
    elif os.path.isdir(s):
        shutil.copytree(s, '/kaggle/working/smartaudit-lab'); break
%cd /kaggle/working/smartaudit-lab

In [ ]:
!pip install -q streamlit anthropic openai accelerate bitsandbytes openpyxl

## 2. Check setup (keys, GPU, data)

In [ ]:
import sys; sys.path.insert(0, '.')
from smartaudit.config import load_secrets_into_env
from smartaudit.registry import list_models
import pandas as pd
print(load_secrets_into_env())
pd.DataFrame(list_models())[['key','tier','ready','status']]

## 3A. Launch the full web app (Streamlit)
Run the cell, then click the `trycloudflare.com` link it prints. The cell keeps running while you use the app.

In [ ]:
from kaggle_launcher import launch
launch()          # or launch(method='ngrok')

## 3B. Or work directly in the notebook (no UI) - good for the assignment
Stop the cell above first (Kaggle runs one cell at a time).

In [ ]:
from smartaudit.data_loader import load_calls, load_sheet
from smartaudit.runner import run_audits
from smartaudit.ui_common_free import sample_ids
from smartaudit import evaluation as ev, report as rp
sheet, calls = load_sheet(), load_calls()
ids = sample_ids(20)
models = ['baseline-lazy', 'baseline-keyword', 'sim-slm', 'sim-llm']      # add 'claude-haiku-5-5', 'gpt-6-luna', 'hf-qwen2.5-3b' ...
res = {m: run_audits(m, ids) for m in models}
ev.compare(res, sheet).round(3)

In [ ]:
# one call, one model: see evidence and rationale
run = run_audits('baseline-keyword', [ids[0]])[0]
print(calls[ids[0]].as_text()); pd.DataFrame([r.__dict__ for r in run.results])

## 4. SLM on the Kaggle GPU
First load takes a few minutes (downloads weights). Use the SLM recipe: one question per call, regional context, compact prompt.

In [ ]:
from smartaudit import slm_tools
slm_ids = sample_ids(8)
r = run_audits('hf-qwen2.5-3b', slm_ids, options=slm_tools.SLM_OPTIONS)
print(ev.compare({'hf-qwen2.5-3b': r}, sheet).round(3).T)

## 5. Cascade: SLM first, LLM for doubtful answers

In [ ]:
key = 'cascade:hf-qwen2.5-3b>claude-haiku-5-5@0.8'     # small>large@confidence-threshold  (append !crit to always escalate critical NOs)
r = run_audits(key, slm_ids)
print(r[0].meta)
ev.compare({key: r}, sheet).round(3).T